In [1]:
"""
Versão 1.4:
Implementação do antigravity para tentar melhorar a inserção de contexto com as tabelas
"""

'\nVersão 1.4:\nImplementação do antigravity para tentar melhorar a inserção de contexto com as tabelas\n'

# Preparação dos Documentos

In [2]:
import pdfplumber
import pandas as pd
import json
import os
import re
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict, Set, Tuple
from statistics import mean
from dotenv import load_dotenv, find_dotenv
import shutil

c:\Users\lucas.andrade_cultin\anaconda3\envs\SLRFIDC_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CHUNK_SIZE = 1000
OFFSET = 200

load_dotenv(find_dotenv())

# Configuração dos Modelos
embeddings_model = OpenAIEmbeddings(model= "text-embedding-3-small")
llm = ChatOpenAI(model = "gpt-4o-mini", temperature= 0, max_tokens= 4096)

# Configuração do VectorStore (ChromaDB)
CHROMADB_PATH   = "G:/Drives compartilhados/RISCO E COMPLIANCE/Relatórios de Risco/Risco/FIDCS/SCRIPTS_RISCO/Projeto IA/Base ChromaDB/"
COLLECTION_NAME = "Regulamentos_V1.4"

vectorstore = Chroma(
    embedding_function = embeddings_model,
    collection_name    = COLLECTION_NAME,
    persist_directory  = CHROMADB_PATH
)

In [4]:
# Função de divisão do documento em chunks de tamanho fixo (Apenas para tabelas)
def split_document(document_text: str) -> List[str]:
    documents = []
    for i in range(0, len(document_text), CHUNK_SIZE):
        start = i
        end = i + CHUNK_SIZE
        if start != 0:
            start = start - OFFSET
            end =  end - OFFSET
        documents.append(document_text[start: end])
    return documents

# Heurística simples para verificar se um caractere faz parte do conteúdo textual
def _is_char_in_line(char: str, line_text: str) -> str:
    return any(c.lower() in line_text.lower() for c in char["text"] if c.strip())

# Extração sensível a contexto de texto dos arquivos PDF
def extract_structured_chunks(pdf_path: str, max_chunk_words: int = 1500) -> List[Dict]:
    chunks = []
    current_chunk = {"chapter": None, "content": ""}

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            chars = page.chars
            if not chars: continue

            # Checa pelo tamanho de fonte superior ao texto da página
            font_sizes = [float(c["size"]) for c in chars]
            avg_font_size = mean(font_sizes)
            big_font_threshold = avg_font_size * 1.2

            lines = page.extract_text().split("\n") if page.extract_text() else []
            for line in lines:
                line_chars = [c for c in chars if _is_char_in_line(c, line)]
                if not line_chars: continue

                # Checagem de fonte e tamanho
                line_font_size = mean([float(c["size"]) for c in line_chars])
                fontnames = set(c["fontname"] for c in line_chars)
                is_bold = any("Bold" in f or "Black" in f or "Heavy" in f for f in fontnames)
                is_big = line_font_size >= big_font_threshold

                # Checagem de alinhamento centralizado
                x0 = min(c["x0"] for c in line_chars)
                x1 = max(c["x1"] for c in line_chars)
                word_center = (x0 + x1) / 2
                page_center = page.width / 2
                is_centered = abs(word_center - page_center) < (page.width * 0.15)

                # Classificação em capítulo ou conteúdo textual com base no título, fonte, tamanho e alinhamento na página
                if (is_big or is_bold or is_centered) and re.match(r"(?i)\s*(cap[ií]tulo)\b", line):
                    if current_chunk["content"].strip():
                        chunks.append(current_chunk.copy())
                    if re.search(r"(?i)cap[ií]tulo", line):     # Início de um novo chunk ao encontrar novo capítulo
                        current_chunk = {"chapter": line.strip(), "content": ""}
                    else:
                        current_chunk["content"] = line.strip()
                    continue

                # Enquanto não encontrar um capítulo, linhas são adicionadas ao conteúdo até um número máximo de palavras (max_chunk_words)
                current_chunk["content"] += " " + line.strip()
                if len(current_chunk["content"].split()) > max_chunk_words:
                    chunks.append({"chapter": current_chunk["chapter"], "content": current_chunk["content"].strip()})
                    current_chunk["content"] = ""

        if current_chunk["content"].strip():
            chunks.append(current_chunk.copy())
    return chunks

# Procura por tabelas no documento e transforma em Dataframe
def extract_tables_with_context(pdf_path: str) -> List[tuple[pd.DataFrame, str]]:
    data_list = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.find_tables()
            for table in tables:
                table_data = page.within_bbox(table.bbox).extract_table()
                if table_data:
                    # Considera a primeira linha como cabeçalho
                    df = pd.DataFrame(table_data[1:], columns=table_data[0])
                    
                    # Delimita uma área acima da tabela como contexto para ser inserida em conjunto no Chunk
                    x0, top, x1, bottom = table.bbox
                    search_top = max(0, top - 150)
                    context_bbox = (0, search_top, page.width, top)
                    context_text = page.within_bbox(context_bbox).extract_text() or ""
                    
                    data_list.append((df, context_text))
    return data_list

# Inserção de tabelas no Banco de Dados
def insert_tables_chromadb(data, source, vectorstore):
    fund = source.split('.pdf', 1)[0]
    
    # Checa pela presença do contexto nos dados recebidos
    context = ""
    if isinstance(data, tuple):
        json_data, context = data
    else:
        json_data = data
        
    # Aplica a divisão de chunks nos dados do JSON
    chunks = split_document(json_data)
    documents_to_add = []
    
    # Construção do chunk
    for idx, chunk in enumerate(chunks):
        metadata = {
            "source": f"{source}_TABLE",
            "fund": fund,
            "original_source_file": source,
            "chunk_index": idx,
            "chapter": "TABLE"  # Tag especial para chunks de tabela
        }

        # Adiciona o contexto ao conteúdo do chunk
        page_content = f"Context: {context}\n\nTable Data: {chunk}"
        documents_to_add.append(Document(page_content=page_content, metadata=metadata))
    
    if documents_to_add:
        vectorstore.add_documents(documents_to_add)

# Inserção de texto no Banco de Dados
def insert_text_chromadb(data, source, vectorstore):
    fund = source.split('.pdf', 1)[0]
    documents_to_add = []
    for idx, chunk in enumerate(data):
        chapter = chunk['chapter'] if chunk['chapter'] else ''
        content = chunk['content']
        
        metadata = {
            "source": f"{source}_{chapter}",
            "fund": fund,
            "chapter": chapter,
            "chunk_index": idx,
            "original_source_file": source
        }
        documents_to_add.append(Document(page_content=content, metadata=metadata))
    
    if documents_to_add:
        vectorstore.add_documents(documents_to_add)


In [5]:
print("Preparando Documentos...")
data_path = 'G:/Drives compartilhados/GESTAO/_Operacional/Planilhas Gestão/Scripts/temp/temp_regulamentos/Avaliação final'

if os.path.exists(data_path):
    documents_names = [f for f in os.listdir(data_path) if f.endswith('.pdf')]
    documents_names_size = len(documents_names)
    
    # Retorna a lista de documentos de fundos já inseridos, ou vazio se não existir
    try:
        stored_data = vectorstore._collection.get(include=["metadatas"])
        available_sources = {
            meta.get("original_source_file", meta.get("fund").split('_')[0]) 
            for meta in stored_data["metadatas"]
            if meta and "source" in meta
        }
    except:
        available_sources = set()

    # Loop pela pasta de regulamentos para inserção no ChromaDB
    for i, document_name in enumerate(documents_names):
        print(f"{i+1}/{documents_names_size}: {document_name}")

        # Segurança contra reinserção
        if document_name in available_sources or document_name.split('.pdf')[0] in available_sources:
            print("Fundo já inserido")
            continue

        # Extração e inserção de texto no ChromaDB
        doc_path = os.path.join(data_path, document_name)
        document_data = extract_structured_chunks(doc_path)
        insert_text_chromadb(document_data, document_name, vectorstore)

        # Extração e inserção de tabelas no ChromaDB
        tables_with_context = extract_tables_with_context(doc_path)
        if tables_with_context:
            for idx, (df, context) in enumerate(tables_with_context):
                
                df = df.loc[:, ~df.columns.duplicated()]

                # Dados de tabelas são convertidos em JSON para melhorar a estrutura de inserção
                json_data = df.to_json(orient= "records", force_ascii=False)
                insert_tables_chromadb((json_data, context), document_name, vectorstore)
else:
    print(f"Diretório {data_path} não encontrado, pulando inserção.")


Preparando Documentos...
1/14: ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS.pdf
Fundo já inserido
2/14: ASTRO SAÚDE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DE RESPONSABILIDADE LIMITADA.pdf
Fundo já inserido
3/14: ATLAS FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS.pdf
Fundo já inserido
4/14: AZURE II FI EM COTAS DE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS - RESPONSABILIDADE LIMITADA.pdf
Fundo já inserido
5/14: BACO FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS.pdf
Fundo já inserido
6/14: CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DO SEGMENTO INDUSTRIAL II - RESP LIMITADA.pdf
Fundo já inserido
7/14: FIDC CONCREDITO.pdf
Fundo já inserido
8/14: FOR-TE FIDC.pdf
Fundo já inserido
9/14: FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS COMERCIAIS VIKING.pdf
Fundo já inserido
10/14: HOPE I FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS MULTISSEGMENTOS.pdf
Fundo já inserido
11/14: MOBILE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS RESPONSABILIDADE LI

# Aplicação do LLM com LangChain

In [17]:
from rapidfuzz import fuzz
from heapq import nlargest

# Realiza uma chamada à LLM para identificar apenas se a pergunta do usuário contém o nome de um FIDC
# Retorna 'FALSE' se nenhum nome for encontrado
def extract_fund_name(query, llm_model):
    template = """
    Identifique o nome do Fundo de Investimento mencionado na pergunta abaixo.
    Retorne APENAS o nome dos fundos identificados, separados por vírgula. 
    Se apenas um fundo for identificado, retorne uma lista mesmo assim
    Se nenhum fundo for mencionado, retorne 'FALSE'.
    
    Pergunta: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm_model | StrOutputParser()
    result = chain.invoke(query)
    
    if result == 'FALSE':
        return None
    return result

# Retorna as top_k melhores correspondências entre o nome do fundo mencionado na pergunta e os fundos disponíveis na base de dados
def detect_document_mention_topk(query_fund_name: str, available_documents: Set[str], top_k: int = 5) -> List[Tuple[str, float]]:

    # Tratamento de entrada vazia
    all_scores = []
    if not available_documents or not query_fund_name:
        return []
        
    print(f"Procurando fundo: {query_fund_name}")
    
    for source in available_documents:
        # Aplica partial_ratio entre o nome_extraído e os fundos disponíveis
        # A função parcial procura pelo melhor alinhamento da String menor na maior
        # Desta forma, a presença de nomes mais descritivos na base não interfere na avaliação
        # Exemplo:
        #       partial_ratio('Itaú', 'Fundo de Investimentos Itaú Cash RF')
        #       resultado: 100.0
        score = fuzz.partial_ratio(query_fund_name.lower(), source.lower())
        if score > 60:
             all_scores.append((source, score))
             
    # Seleciona as 5 correspondencias de maior score
    top_matches = nlargest(top_k, all_scores, key=lambda item: item[1])
    print(f"Top matches for '{query_fund_name}': {top_matches}")
    return top_matches


def ask_llm(query, vectorstore, llm_model):

    # Listagem dos nomes de fundos disponíveis no sistema
    # ---------------------------------------------------------------------------------------------------#
    try:
        stored_data = vectorstore._collection.get(include=["metadatas"])
        available_funds = {meta["fund"] for meta in stored_data["metadatas"] if meta and "fund" in meta}
    except:
        available_funds = set()
    # ---------------------------------------------------------------------------------------------------#

    # Extração da lista de possíveis nomes de fundos mencionados na pergunta
    extracted_name = extract_fund_name(query, llm_model)
    
    # Procura no banco pelo fundo extraído
    top_k_funds = []
    if extracted_name:
        for fund in extracted_name.split(','):
            top_k_funds.append([item[0] for item in detect_document_mention_topk(fund, available_funds)][0])

    # Preparação do Retriever e seus argumentos
    # Aplica o nome do fundo como filtro, caso algum seja mencionado 
    # -----------------------------------------------------------------#
    search_kwargs = {"k": 5}
    if top_k_funds:
        search_kwargs["filter"] = {"fund": {"$in": top_k_funds}}
        print(f"Filtering by funds: {top_k_funds}")
    else:
        print("No specific fund detected. Searching all documents.")

    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    # -----------------------------------------------------------------#
    
    # Construção da Chain
    template = """
    Você é um assistente especializado em responder perguntas sobre fundos de investimento.
    Use o seguinte contexto para responder à pergunta.
    Se não souber a resposta com base no contexto, diga que não encontrou a resposta nos documentos disponíveis. 
    Contexto: {context}
    
    Pergunta: {question}
    """
    prompt = ChatPromptTemplate.from_template(template)
    
    def format_docs(docs):
        return "\n\n".join([d.page_content for d in docs])
    
    map_setup = RunnableParallel(
        {
          "context_docs" : retriever,
          "question"     : RunnablePassthrough()
        }
    )

    chain = (
        map_setup
        | {
            "context"  : lambda x: format_docs(x["context_docs"]),
            "question" : lambda x: x["question"],
            "docs"     : lambda x: x["context_docs"]
          }
        | {
            "output" : prompt | llm_model | StrOutputParser(),
            "sources" : lambda x: [
                {
                    "index"       : i,
                    "chunk_index" : doc.metadata.get("chunk_index", "N/A"),
                    "content"     : doc.page_content,
                    "metadata"    : doc.metadata
                }
                for i, doc in enumerate(x["docs"])
            ]
        }
    )
   
    return chain.invoke(query)


In [11]:
question = """
Me diga quem são o gestor e o administrador do seguinte fundo de investimento: FIDC Artesanal Feeder
"""
res = extract_fund_name(question, llm)

In [18]:
question = """
Me diga quem são o gestor e o administrador do seguinte fundo de investimento: FIDC Artesanal Feeder
"""

answer = ask_llm(question, vectorstore, llm)

print("Resposta: ", answer['output'])
print("Fontes: ")
for source in answer['sources']:
    print(f"Chunk [{source['index']}] - ID Original: {source['chunk_index']} - Conteúdo: {source['content']}")

Procurando fundo: FIDC Artesanal Feeder
Top matches for 'FIDC Artesanal Feeder': [('ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS', 86.48648648648648), ('FIDC CONCREDITO', 60.86956521739131)]
Filtering by funds: ['ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS']
Resposta:  O gestor do FIDC Artesanal Feeder é a ARTESANAL INVESTIMENTOS LTDA, e o administrador é a GENIAL INVESTIMENTOS CORRETORA DE VALORES MOBILIÁRIOS S.A.
Fontes: 
Chunk [0] - ID Original: 0 - Conteúdo: Docusign E nvelope ID: D743CFDD-1438-4043-9BE6-EF3CCBC1B58F REGULAMENTO DO ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS São Paulo, 6 de dezembro de 2024. Docusign Envelope ID: D743CFDD-1438-4043-9BE6-EF3CCBC1B58F REGULAMENTO DO ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS 1. CARACTERÍSTICAS DO FUNDO 1.1. O ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS, disciplinado pela Resolução CMN 2.907, pela Lei nº 10.406, de 10 de janei

In [ ]:
def delete_fund_chunks(fund_name, vectorstore):
    print(f"Deleting chunks for fund: {fund_name}")
    try:
        # Get IDs to confirm (optional, but good for logging)
        # Chroma/LangChain delete might need IDs or filter.
        # LangChain Chroma `delete` supports `where` filter? 
        # Checking logic: vectorstore.delete(ids=...) is standard.
        # We need to find IDs first using the filter.
        
        results = vectorstore._collection.get(where={"fund": fund_name})
        ids_to_delete = results['ids']
        
        if ids_to_delete:
            print(f"Found {len(ids_to_delete)} chunks. Deleting...")
            vectorstore.delete(ids=ids_to_delete)
            print("Deletion complete.")
        else:
            print("No chunks found for this fund.")
            
    except Exception as e:
        print(f"Error deleting chunks: {e}")

# Usage Example:
# delete_fund_chunks("CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DO SEGMENTO INDUSTRIAL II - RESP LIMITADA", vectorstore)

Deleting chunks for fund: CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DO SEGMENTO INDUSTRIAL II - RESP LIMITADA
Found 76 chunks. Deleting...
Deletion complete.


In [5]:
def get_fund_chunks(fund_name, vectorstore):
    """
    Retorna todos os chunks de um determinado fundo da vectorstore.
    """
    print(f"Buscando chunks para o fundo: {fund_name}")
    try:
        # Busca direta na coleção do ChromaDB filtrando pelo metadata "fund"
        results = vectorstore._collection.get(where={"fund": fund_name})
        
        chunks = []
        if results['ids']:
            for i in range(len(results['ids'])):
                chunk = {
                    'id': results['ids'][i],
                    'content': results['documents'][i] if results['documents'] else "",
                    'metadata': results['metadatas'][i] if results['metadatas'] else {}
                }
                chunks.append(chunk)
            print(f"Encontrados {len(chunks)} chunks.")
            return chunks
        else:
            print("Nenhum chunk encontrado para este fundo.")
            return []
            
    except Exception as e:
        print(f"Erro ao buscar chunks: {e}")
        return []

In [7]:
stored_data = vectorstore._collection.get(include=["metadatas"])
available_funds = {meta["fund"] for meta in stored_data["metadatas"] if meta and "fund" in meta}

In [ ]:
available_funds
# ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS : 4, 55
# ASTRO SAÚDE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DE RESPONSABILIDADE LIMITADA : 21, 47
# ATLAS FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS : 27, 104
# AZURE II FI EM COTAS DE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS - RESPONSABILIDADE LIMITADA : 43, 65
# BACO FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS: 37, 55
# CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DO SEGMENTO INDUSTRIAL II - RESP LIMITADA: 44, 60
# FIDC CONCREDITO : 35, 73
# FOR-TE FIDC : 29, 77
# FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS COMERCIAIS VIKING : 43 , 96
# HOPE I FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS MULTISSEGMENTOS : 19, 79
# PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS NÃO PADRONIZADOS MULTISSETORIAL : 16, 88
# Regulamento Poupacred II : 48 , 77

{'ARTESANAL FEEDER ST FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS',
 'ASTRO SAÚDE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DE RESPONSABILIDADE LIMITADA',
 'ATLAS FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS',
 'AZURE II FI EM COTAS DE FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS - RESPONSABILIDADE LIMITADA',
 'BACO FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS',
 'CATERPILLAR FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS DO SEGMENTO INDUSTRIAL II - RESP LIMITADA',
 'FIDC CONCREDITO',
 'FOR-TE FIDC',
 'FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS COMERCIAIS VIKING',
 'HOPE I FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS MULTISSEGMENTOS',
 'PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS NÃO PADRONIZADOS MULTISSETORIAL',
 'Regulamento FIF CAM SOBERANO CASH',
 'Regulamento Poupacred II'}

In [20]:
get_fund_chunks("PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS NÃO PADRONIZADOS MULTISSETORIAL", vectorstore)

Buscando chunks para o fundo: PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS NÃO PADRONIZADOS MULTISSETORIAL
Encontrados 16 chunks.


[{'id': 'PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITORIOS NÃO PADRONIZADOS MULTISSETORIAL.pdf_chunk_0',
  'content': 'REGULAMENTO DO PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS 03 de fevereiro de 2025 1 b4666448-0318-464a-a479-e421e7dae232 GLOSSÁRIO DOS PRINCIPAIS TERMOS E EXPRESSÕES UTILIZADOS NO REGULAMENTO DO PUMA FUNDO DE INVESTIMENTO EM DIREITOS CREDITÓRIOS Definições. Os termos e expressões utilizados neste Regulamento, quando iniciados por letra maiúscula, têm o significado a eles atribuídos no Glossário abaixo. Além disso, (i) sempre que exigido pelo contexto, as definições contidas neste Regulamento aplicar- se-ão tanto no singular quanto no plural e o gênero masculino incluirá o feminino e vice- versa; (ii) referências a qualquer documento ou outros instrumentos incluem todas as suas alterações, substituições, consolidações e respectivas complementações, salvo se expressamente disposto de forma diferente; (iii) referências a disposições legais serão interpretadas com